<div style="background-color:#f5f3ff; border-radius:8px; padding:12px; text-align:center;">

# **PRACTICA 2: Juego de Tronos · VISUALIZACIÓN DE DATOS**

</div>

*Visualización de Juego de Tronos*

---

**Grupo:** G-7312  
**Número de pareja:** 01  
**Miembros:**  
- Leire Bernárdez Vázquez  
- Carmen Reiné Rueda


---

### **CONFIGURACIONES PREVIAS**

<div style="background-color:#f5f3ff; color:#6a0dad; padding:10px; border-radius:5px;">
Importaciones
</div>

In [35]:
import numpy as np
import pandas as pd
import json
import os
import glob
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import networkx as nx       
import geopandas as gpd      
from folium.plugins import MarkerCluster
from collections import Counter

In [36]:
# === CARGA DE LOS DATOS ===

# Rutas base
ruta_data = r"C:\Users\carme\OneDrive - UAM\TERCERO\PRIMER CUATRI\VD\PRACTICAS\VD_G7312_P02\game_of_thrones\data"
ruta_map = r"C:\Users\carme\OneDrive - UAM\TERCERO\PRIMER CUATRI\VD\PRACTICAS\VD_G7312_P02\GOT-Map\GOT-Map"

# --- Carga de archivos JSON principales ---
with open(os.path.join(ruta_data, "characters.json"), "r", encoding="utf-8") as f:
    characters = json.load(f)

with open(os.path.join(ruta_data, "episodes.json"), "r", encoding="utf-8") as f:
    episodes = json.load(f)

with open(os.path.join(ruta_data, "locations.json"), "r", encoding="utf-8") as f:
    locations = json.load(f)

# --- Carga de archivos GeoJSON (mapas) ---
land = gpd.read_file(os.path.join(ruta_map, "land.geojson"))
places = gpd.read_file(os.path.join(ruta_map, "places.geojson"))

---


### **EJERCICIOS**



### <span >**Parte 1: Visualización de grafos con NetworkX**

En esta primera parte trabajaremos con las relaciones existentes entre los personajes de la serie *Game of Thrones* utilizando la librería **NetworkX**.  
El objetivo es construir el grafo de relaciones, obtener métricas y justificar las decisiones de diseño adoptadas (tipo de grafo, relaciones representadas, tamaño y color de nodos, y layout empleado).




<div style="background-color:#f5f3ff; color:#6a0dad; padding:10px; border-radius:5px;">
Apartado 1: Construcción del grafo de relaciones
</div>



A partir del fichero `characters.json`, construiremos un grafo donde los nodos representen personajes y las aristas indiquen algún tipo de relación entre ellos (parentesco, conflicto, etc.).


Antes de construir el grafo, realizamos una exploración básica del fichero para conocer cuántos personajes contiene, si hay registros sin nombre o duplicados, y qué información incluye. También miraremos todas los campos presentes en el dataset para distinguir cuáles son atributos descriptivos (nombre, actor, casa, etc.) y cuáles representan **relaciones entre personajes**.


In [37]:
data_chars = characters["characters"]
print(f"Personajes totales (sin limpieza): {len(data_chars)}")

# ¿Hay personajes SIN nombre?
sin_nombre = [i for i,c in enumerate(data_chars) if not c.get("characterName")]
print("\nSin nombre:", len(sin_nombre))

# ¿Hay nombres duplicados?
contador = Counter([c["characterName"] for c in data_chars if c.get("characterName")])
duplicados = [n for n,c in contador.items() if c>1]
print("\nDuplicados:", len(duplicados))
if duplicados:
    print(duplicados[:])  
    
# Todas los campos posibles en el JSON
todos_los_campos = sorted({k for c in data_chars for k in c.keys()})
print("\nTotal de campos en el JSON:", len(todos_los_campos))
print(todos_los_campos)


Personajes totales (sin limpieza): 389

Sin nombre: 0

Duplicados: 15
['Goldcloak', 'Handmaid', 'High Septon', 'Lannister Captain', 'Little Bird', 'Musician #1', 'Musician #2', 'Musician #3', "Night's Watch Officer", "Night's Watchman", "Night's Watchman #2", 'Red Priestess', 'Septon', 'Stark Guard', 'White Walker']

Total de campos en el JSON: 25
['abducted', 'abductedBy', 'actorLink', 'actorName', 'actors', 'allies', 'characterImageFull', 'characterImageThumb', 'characterLink', 'characterName', 'guardedBy', 'guardianOf', 'houseName', 'killed', 'killedBy', 'kingsguard', 'marriedEngaged', 'nickname', 'parentOf', 'parents', 'royal', 'servedBy', 'serves', 'sibling', 'siblings']


Estos personajes duplicados no son errores, sino roles genéricos o figurantes sin relevancia narrativa.  
Posteriormente se eliminarán para simplificar el grafo y centrarnos en los personajes principales.  


**1. Explora los tipos de relaciones disponibles en el fichero y selecciona al menos tres ipos diferentes para incluir en el grafo.**

El dataset contiene 25 campos diferentes, hemos seleccionado únicamente aquellos campos que representan vínculos entre personajes (por ejemplo: parents, parentOf, siblings, killed, killedBy, marriedEngaged, etc.).  
El resto de atributos descriptivos (como characterName, actorName, houseName, etc.) se mantendrán solo si aportan información adicional en el grafo.

In [38]:
# Campos que consideramos "de relación" según el JSON real
relaciones_validas = {
    "parents", "parentOf", "siblings", "sibling",
    "killed", "killedBy",
    "marriedEngaged",
    "servedBy", "serves",
    "guardianOf", "guardedBy",
    "allies", "abducted", "abductedBy"
}

# Las que son realmente relaciones
relaciones_personajes = sorted({k for k in todos_los_campos if k in relaciones_validas})
print("\nSolo las campos que son relaciones válidas:", len(relaciones_personajes))
print("Tipos de relaciones presentes:", relaciones_personajes)




Solo las campos que son relaciones válidas: 14
Tipos de relaciones presentes: ['abducted', 'abductedBy', 'allies', 'guardedBy', 'guardianOf', 'killed', 'killedBy', 'marriedEngaged', 'parentOf', 'parents', 'servedBy', 'serves', 'sibling', 'siblings']


**Limpieza del dataset: eliminación de duplicados y columnas no necesarias**

Antes de construir el grafo, realizamos una limpieza del dataset para eliminar:

- Personajes genéricos o duplicados, que no aportan relaciones significativas.
- Columnas que no son relevantes para la práctica, como los enlaces de actor o las imágenes, ya que no influyen en la estructura del grafo.

Dejaremos únicamente los campos necesarios para representar nodos y relaciones válidas .

In [39]:
# Personajes genéricos o repetidos
nombres_duplicados = 'Goldcloak', 'Handmaid', 'High Septon', 'Lannister Captain', 'Little Bird', 'Musician #1', 'Musician #2', 'Musician #3', "Night's Watch Officer", "Night's Watchman", "Night's Watchman #2", 'Red Priestess', 'Septon', 'Stark Guard', 'White Walker'

# Campos relevantes: nombre, casa, estatus real y relaciones útiles
campos = ['characterName', 'houseName', 'royal', "kingsguard", "nickname"]

campos_utiles = campos + relaciones_personajes

# Crear nuevo dataset filtrado
data_chars_limpio = []

for c in data_chars:
    nombre = c.get("characterName")
    # Saltamos los personajes duplicados o sin nombre
    if not nombre or nombre in nombres_duplicados:
        continue

    # Conservamos solo las columnas útiles (atributos + relaciones)
    limpio = {k: c[k] for k in campos_utiles if k in c}
    data_chars_limpio.append(limpio)

print(f"Personajes antes de limpiar: {len(data_chars)}")
print(f"Personajes después de limpiar: {len(data_chars_limpio)}")

# Comprobamos algunos campos del primer registro limpio
print("\nEjemplo de estructura limpia:")
for k, v in data_chars_limpio[1].items():
    print(f"- {k}: {v }")

# === GUARDAR EL NUEVO JSON LIMPIO ===
ruta_salida = os.path.join(ruta_data, "characters_clean.json")

with open(ruta_salida, "w", encoding="utf-8") as f:
    json.dump(data_chars_limpio, f, ensure_ascii=False, indent=3)

print(f"\nArchivo limpio guardado correctamente en:\n{ruta_salida}")
print(f"Total de personajes guardados: {len(data_chars_limpio)}")


Personajes antes de limpiar: 389
Personajes después de limpiar: 355

Ejemplo de estructura limpia:
- characterName: Aegon Targaryen
- houseName: Targaryen
- royal: True
- killedBy: ['Gregor Clegane']
- parents: ['Elia Martell', 'Rhaegar Targaryen']
- siblings: ['Rhaenys Targaryen', 'Jon Snow']

Archivo limpio guardado correctamente en:
C:\Users\carme\OneDrive - UAM\TERCERO\PRIMER CUATRI\VD\PRACTICAS\VD_G7312_P02\game_of_thrones\data\characters_clean.json
Total de personajes guardados: 355


**2. Decide qué relaciones deben representarse como dirigidas y cuáles como no 
dirigidas**

In [40]:
# === CARGA DEL JSON LIMPIO ===
ruta_clean = os.path.join(ruta_data, "characters_clean.json")
with open(ruta_clean, "r", encoding="utf-8") as f:
    personajes = json.load(f)

# === DETECTAMOS RELACIONES PRESENTES Y SU FRECUENCIA ===
rel_counter = Counter()

for c in personajes:
    for k, v in c.items():
        if isinstance(v, list): 
            rel_counter[k] += len(v)

print("\nTipos de relaciones encontradas y su frecuencia total:")
for rel, n in rel_counter.most_common():
    print(f"- {rel}: {n}")

# Cuántos personajes tienen al menos una relación
personajes_con_rel = sum(
    any(isinstance(v, list) and len(v) > 0 for v in c.values())
    for c in personajes
)
print(f"\nPersonajes con al menos una relación: {personajes_con_rel}")



Tipos de relaciones encontradas y su frecuencia total:
- killed: 221
- killedBy: 203
- siblings: 134
- parents: 81
- parentOf: 78
- marriedEngaged: 55
- serves: 22
- guardianOf: 15
- guardedBy: 12
- servedBy: 10
- houseName: 10
- allies: 8
- sibling: 2
- abductedBy: 1
- abducted: 1

Personajes con al menos una relación: 226


A partir del análisis de frecuencias realizado, observamos que hay relaciones principales y otras aparecen con poca frecuencia y no aportan una estructura suficientemente rica para el grafo general.

Basándonos en estos resultados y en los ejemplos comentados en clase, hemos decidido centrarnos en **tres tipos de relaciones significativas**:
- **killed** → relación dirigida, representa un conflicto donde un personaje mata a otro (A → B).  
- **siblings** → relación no dirigida, representa vínculos de hermandad mutua.  
- **parents** → relación no dirigida, representa lazos familiares entre padres e hijos sin sentido de dirección.

De este modo, construiremos un **grafo mixto**, ya que combina relaciones **dirigidas** (de conflicto)  y **no dirigidas** (de tipo familiar o de parentesco), reflejando tanto los enfrentamientos como los lazos de sangre entre los personajes.



**3. Crea el grafo en NetworkX, añadiendo atributos a las aristas que indiquen el tipo de 
relación.**